# Building an Agentic RAG System

In this exercise, you will build an Agentic RAG (Retrieval-Augmented Generation) system that 
combines the power of AI agents with traditional RAG pipelines. You'll create an agent that 
can decide when and how to retrieve information from different sources, including vector 
databases, web search, and other tools.


## Challenge

Your challenge is to create an Agentic RAG system that can:

- Build a RAG pipeline as a tool that can be used by the agent
- Create an agent that can decide which tool to use based on the query
- Handle different types of queries intelligently
- Combine information from multiple sources when needed


## Setup
First, let's import the necessary libraries:

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [2]:
import os
from typing import List
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import BaseMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG

In [3]:
load_dotenv()

True

In [4]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Load data to Vector DB

In [5]:
db = VectorStoreManager(OPENAI_API_KEY)
db

VectorStoreManager():<chromadb.api.client.Client object at 0x748ebaf217f0>

In [6]:
loader_service = CorpusLoaderService(db)

In [7]:
rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

In [9]:
games_market_rag = RAG(
    llm=rag_llm,
    vector_store = loader_service.load_pdf(
        store_name="game_market",
        pdf_path="TheGamingIndustry2024.pdf"
    )
)

VectorStore `game_market` ready!
Pages from `TheGamingIndustry2024.pdf` added!


In [10]:
result:Run = games_market_rag.invoke(
    "What's the  state of virtual reality"
)
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
The state of virtual reality (VR) in 2024 is characterized by significant advancements and increasing integration into the gaming industry. VR technology allows gamers to immerse themselves in expansive digital worlds, offering realistic and interactive experiences. Games like "Half-Life: Alyx" exemplify this trend by providing high-fidelity graphics, intuitive controls, and immersive audio, enhancing player engagement. Additionally, VR is being utilized beyond entertainment, with applications in education, cognitive therapy, physical rehabilitation, and mental health, indicating its transformative potential in various fields. Overall, VR is evolving as a powerful tool that reshapes how players interact with games and the world around them.


In [11]:
electric_vehicles_rag = RAG(
    llm=rag_llm,
    vector_store = loader_service.load_pdf(
        store_name="electric_vehicles",
        pdf_path="GlobalEVOutlook2025.pdf"
    )
)

VectorStore `electric_vehicles` ready!
Pages from `GlobalEVOutlook2025.pdf` added!


In [12]:
result:Run = electric_vehicles_rag.invoke("What was the number of electric car sales and their market share in Brazil in 2024?")
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
In 2024, Brazil recorded nearly 125,000 electric car sales, which represented a market share of 6.5%. This was more than double the number of sales from 2023.


## Tools

In a simple form, Agentic RAG can act like a router, choosing between multiple external sources to retrieve relevant information. These sources aren't limited to databases, they can also include tools like web search or APIs for services such as Slack or email.

In this case it will choose between two collections.

In [13]:
# Define a tool that returns result.get_final_state()["answer"]

@tool
def search_global_ev_collection(query):
    """
    Search the vector database for relevant information.
    
    Source: Global EV Outlook 2025
    Publisher: International Energy Agency
    Release: May 2025
    ```
        The Global EV Outlook is an annual publication that reports on recent
        developments in electric mobility around the world. It is developed with the support
        of members of the Electric Vehicles Initiative (EVI).
    ```

    args:
        query (str): Search query
    """
    result:Run = electric_vehicles_rag.invoke(query)
    return result.get_final_state()["answer"]

In [14]:
# TODO: Define a tool that returns result.get_final_state()["answer"]
# DONOT Forget about defining the tool docstrings
@tool
def search_games_market_report_collection(query):
    """    
    Search the vector database for relevant information.
    
    Source: The Gaming Industry in 2024 - Trends, Technologies & Predictions
    Publisher: Ixie Gaming
    Release: 2024
    ```
        The gaming industry, on the brink of transformative change due to technological
        advancements, is redefining entertainment and social interaction with
        immersive, personalized, and interactive experiences.
    ```

    args:
        query (str): Search query
    """
    result:Run = games_market_rag.invoke(query)
    return result.get_final_state()["answer"]

In [15]:
# TODO: Add the tools you have defined and the instructions to your agent
agentic_rag = Agent(
    model_name="gpt-4o-mini",
    tools=[search_global_ev_collection, search_games_market_report_collection],
    instructions=(
        "You are an Agentic RAG assistant that can intelligently decide which tools to use "
        "to answer user questions. Reason about about the response, change the query and call the tool again if needed "
        "in order to get better results. Always explain your reasoning for tool selection and provide comprehensive answers."
    )
)

In [16]:
def print_messages(messages: List[BaseMessage]):
    for m in messages: 
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run

In [17]:
run_1 = agentic_rag.invoke(
    query="Who won the 2025 Oscar for International Movie?", 
    session_id="oscar",
)

print("\nMessages from run 1:")
messages = run_1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an Agentic RAG assistant that can intelligently decide which tools to use to answer user questions. Reason about about the response, change the query and call the tool again if needed in order to 

In [18]:
run_2 = agentic_rag.invoke(
    query= (
        "Which two countries accounted for most of the electric car exports from " 
        "the Asia Pacific region (excluding China) in 2024?"
    ),
    session_id="electric_car",
)

print("\nMessages from run 2:")
messages = run_2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are an Agentic RAG assistant that can intelligently decide which tools to use to answer user questions. Reason about about the response, change the query and call the tool again if needed in order to get better results. Always explain your reasoning for tool selection and provide comprehensive answers., tool_calls = None)
 -> (role = user, content = Which two countries accounted for most of the electric car exports from the Asia Pacific region (excluding China) in 2024?, tool_calls =

In [19]:
run_3 = agentic_rag.invoke(
    query= (
        "Why is generative AI seen more as an accelerator than a replacement in game development?"
    ),
    session_id="games",
)

print("\nMessages from run 3:")
messages = run_3.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachin